# Notebook 3: Radar/Lidar Point Cloud Attacks

Demonstrate all 6 point cloud attack types on Radar and Lidar.

**Attacks:** Ghost Injection, Cluster Split, Cluster Merge, Point Suppression, Noise Floor, Random Perturbation

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, get_ground_truth, get_ownship

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
from attacks.radar_lidar_attacks import PointCloudAttacker, PointCloudAttackType

## 3.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)

lidar = detections[1]
radar = detections[2]

print('Lidar:', len(lidar), 'detections')
print('Radar:', len(radar), 'detections')

## 3.2 Initialize Attacker

In [ ]:
attacker = PointCloudAttacker()
print('Available attacks:', [a.name for a in PointCloudAttackType])

## 3.3 Run Attacks on Lidar

In [ ]:
attacks = [
    PointCloudAttackType.GHOST_INJECTION,
    PointCloudAttackType.CLUSTER_SPLIT,
    PointCloudAttackType.CLUSTER_MERGE,
    PointCloudAttackType.POINT_SUPPRESSION,
    PointCloudAttackType.NOISE_FLOOR,
    PointCloudAttackType.RANDOM_PERTURBATION
]

lidar_results = {}
for attack in attacks:
    attacked = attacker.attack_detections(lidar.copy(), attack, sensor_id=1)
    lidar_results[attack.name] = attacked
    print(attack.name + ':', len(attacked), 'detections (delta:', len(attacked) - len(lidar), ')')

## 3.4 Visualize Point Cloud Attacks

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (attack_name, attacked_df) in enumerate(lidar_results.items()):
    ax = axes[idx]
    ax.scatter(lidar['x_piren'], lidar['y_piren'], s=3, alpha=0.3, c='blue', label='Benign')
    ax.scatter(attacked_df['x_piren'], attacked_df['y_piren'], s=3, alpha=0.3, c='red', label='Attacked')
    ax.set_title(attack_name + '\n' + str(len(attacked_df)) + ' detections')
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.legend()
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Lidar Point Cloud Attacks', fontsize=14)
plt.tight_layout()
plt.show()

## 3.5 Radar vs Lidar Attack Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for col, attack in enumerate(attacks[:3]):
    att_lidar = attacker.attack_detections(lidar.copy(), attack, sensor_id=1)
    ax = axes[0, col]
    ax.scatter(lidar['x_piren'], lidar['y_piren'], s=3, alpha=0.3, c='blue')
    ax.scatter(att_lidar['x_piren'], att_lidar['y_piren'], s=3, alpha=0.3, c='red')
    ax.set_title('Lidar - ' + attack.name)
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)
    ax.set_aspect('equal')
    
    att_radar = attacker.attack_detections(radar.copy(), attack, sensor_id=2)
    ax = axes[1, col]
    ax.scatter(radar['x_piren'], radar['y_piren'], s=3, alpha=0.3, c='blue')
    ax.scatter(att_radar['x_piren'], att_radar['y_piren'], s=3, alpha=0.3, c='red')
    ax.set_title('Radar - ' + attack.name)
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Point Cloud Attacks: Lidar vs Radar', fontsize=14)
plt.tight_layout()
plt.show()